# Validation — `kappa-lora-halve-params`

**What this measures:** `num_trainable_params` is the field the repo's own MetaMathQA harness emits and directly quantifies this PR's mechanism: with the experiment config now mirroring the published `lora--llama-3.2-3B-rank32` row exactly (r=32 over target_modules ['v_proj','q_proj'] → (32·(3072+3072) + 32·(3072+1024)) × 28 layers = 9,175,040, the row's own value) and changing only what this PR introduces (`condition_number_top_fraction=0.5`), LoRA is injected into just the top half of the 56 matched modules, so the drop below the row value measures the claimed parameter halving on the maintainers' own protocol — guarded by `test_accuracy` floored at the row's own 0.49052312357846856 so fit cannot regress below the row it is compared against.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`ebbba726a323`](https://github.com/mayorquinmachines/peft/commit/ebbba726a323102f34d01a98892b938e023f8841)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "ebbba726a323102f34d01a98892b938e023f8841"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/kappa-lora/llama-3.2-3B-rank32` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/kappa-lora/llama-3.2-3B-rank32/adapter_config.json`:

```json
{"peft_type": "LORA", "task_type": "CAUSAL_LM", "base_model_name_or_path": "meta-llama/Llama-3.2-3B", "inference_mode": false, "r": 32, "target_modules": ["v_proj", "q_proj"], "condition_number_top_fraction": 0.5}
```

In [ ]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/kappa-lora/llama-3.2-3B-rank32/adapter_config.json")).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/kappa-lora/llama-3.2-3B-rank32` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/kappa-lora/llama-3.2-3B-rank32/*/")) or ["experiments/kappa-lora/llama-3.2-3B-rank32"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `results/kappa-lora--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["results/kappa-lora--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = []
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = []

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    print(f"{c['metric']:<28}{str(v):>16}{str(c['baseline']):>16}  {c['direction']} {t}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
benchmarks:
  - name: kappa-lora-halve-params
    suite:
      harness:
        notebook: .remyx/validations/kappa-lora-halve-params.ipynb
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/kappa-lora/llama-3.2-3B-rank32
        results_glob: method_comparison/MetaMathQA/results/kappa-lora--*.json
        method: lora
      scorer: num_trainable_params
      metrics:
        - name: num_trainable_params
          direction: min
          # derivation (config now mirrors the published row's own configuration, r=32 over target_modules ['v_proj','q_proj']): per layer = 32*(3072+3072) [q_proj] + 32*(3072+1024) [v_proj] = 196,608 + 131,072 = 327,680; x 28 layers = 9,175,040 = exactly the row's num_trainable_params (9175040.0). condition_number_top_fraction=0.5 keeps the top ceil(56*0.5)=28 of the 56 matched modules; each selected module costs 196,608 (q_proj) or 131,072 (v_proj), so any correct top-28 selection lands in [3,670,016, 5,505,024]. Threshold = claim factor 0.5 x worst-case module-size skew (max per-module cost / mean = 196,608/163,840 = 1.2) = 0.6 x 9,175,040 = 5,505,024 — the tightest bound no correct top-half selection can exceed, which the published row (9,175,040) fails, certifying >= 40% fewer trainable params (the halving claim up to module-size skew).
          threshold: 5505024
          role: target
        - name: test_accuracy
          direction: max
          # floor moved onto the published row's own test_accuracy (0.49052312357846856, read from lora--llama-3.2-3B-rank32.json), which the baseline row itself meets at equality: the guardrail now bounds regression from the row this experiment is compared against, so a kappa-LoRA run that "matches standard LoRA accuracy" must not land below it, while degenerate spectral targeting or dropped modules collapse far below; the prior 0.5 bar was a bar the row itself fails.
          threshold: 0.49052312357846856
          role: guardrail
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 9175040
        test_accuracy: 0.49052312357846856
    policy:
      guardrail_veto: true
    held_constant:
      - "base model: meta-llama/Llama-3.2-3B, same weights as the published lora row"
      - "LoRA rank r=32 over target_modules ['v_proj', 'q_proj'] — exactly the configuration that produced the published lora--llama-3.2-3B-rank32 row; the only difference is this PR's new condition_number_top_fraction=0.5"
      - "training protocol from the harness default_training_params.json (seed, lr, batch size, steps) unchanged"
      - "same MetaMathQA test split and accuracy metric as the published corpus"
    avoid:
      - "unpinned base-model revision"
      - "overriding default_training_params.json"
      - "wall-clock gating across arms"
      - "changing target_modules or r relative to the published row — that breaks like-for-like comparison with the baseline row"
    compute:
      tier: gpu
      # one arm = 3B bf16 LoRA fine-tune on the harness's own MetaMathQA protocol (~5k-25k steps at ~1.5-2 s/step on a single A100/H100, protocol peaks >22 GB VRAM per the suite docs) plus test-set eval; 12 h with headroom.
      timeout_s: 43200
    provenance:
      num_trainable_params: "user_guidance (row value 9175040.0 as recorded in lora--llama-3.2-3B-rank32.json) + maintainer_comment:@mayorquinmachines"
      test_accuracy: "user_guidance (row value 0.49052312357846856 as recorded in lora--llama-3.2-3B-rank32.json) + maintainer_comment:@mayorquinmachines"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      experiments: "user_guidance: mirror the row's configuration (r=32, target_modules ['v_proj', 'q_proj']) and add only this PR's condition_number_top_fraction=0.5"
      baseline: "published corpus: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json (num_trainable_params = 9175040.0, test_accuracy = 0.49052312357846856, read from the row)"
      held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md"
```